### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="immoscout_german_house_prices",
    dataset_year="2019",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Kaggle", # original source Kaggle, download source OpenML.
    original_dataset_source_download_link="https://www.openml.org/d/43342",
    download_description="""
We download the data from OpenML (original Kaggle dataset has been deleted).

Also available at:
- https://www.kaggle.com/code/shritech1404/german-housing-price-prediction (reference only, original dataset deleted)

wget https://www.openml.org/data/download/22102167/dataset -O dataset.arff && mkdir -p local-data-warehouse/immoscout_german_house_prices && mv dataset.arff local-data-warehouse/immoscout_german_house_prices/
""",
    # References
    academic_reference_bibtex=r"""@misc{OpenML43342Dataset,
  author = {{OpenML}},
  title  = {ImmoScout24 OpenML Dataset 43342},
  year   = {2023},
  howpublished = {\url{https://www.openml.org/d/43342}},
  note   = {OpenML dataset}
}
@misc{Shritech2019GermanHousingPricePrediction,
  author = {shritech1404},
  title  = {German Housing Price Prediction},
  year   = {2019},
  howpublished = {\url{https://www.kaggle.com/code/shritech1404/german-housing-price-prediction}},
  note   = {Kaggle notebook}
}
""",
    academic_reference_bibtex_key="Shritech2019GermanHousingPricePrediction,OpenML43342Dataset",
    license="CC BY-NC-SA 4.0", # From OpenML, no idea about the original license as it was on Kaggle.
    data_tags=["IID"],
    curation_comments="""
We treat this as an IID time-independent task, because the prices is not sold prices but instead the offer price, so the snapshot here represents a task where someone would want to predict the price they should offer for their house given other houses that are currently on the market. Thus, it seems like a real IID task where someone would want to build a model to know how much they should offer their houses at in the current market (with the limitation that past trends are ignored)

- We log scale it, as is appropriate for house price prediction tasks. We do not normalize by lot or living space, as both houses with a lot of land and small living space and houses with a lot of living space and small land are interesting to predict the price of, and normalization would bias towards one of them, as both cannot be homogenised.
- We drop Free_of_Relation as it indicates a (mostly) price-independent variable; showing logistic information. Note that this column also indicates that the houses are most likely from the same time, further justifying our use case of an IID task.
- We drop all cases where the price is less than 10k (most of which have a price of 0) as we think this might be a data error, or a case where the price is put to 0 to be marked as "price on request" (which is common in house listings).
- We drop all duplicates as these are most likely repeated entries given the feature space.
- We drop all rows where the living space is less than 10 (in Germany, this would likely not be enough living space for a real house, and thus might be a data error.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="LogPrice",
    problem_type="regression",
    objective_metric_name="rmse",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np
import arff

with open(dataset_mold.path / "dataset.arff", "r") as f:
    data = arff.load(f)
df = pd.DataFrame(data["data"], columns=[attr[0] for attr in data["attributes"]])

# Data realistic drops
df = df[df["Price"] >=10000]
df = df[df["Living_space"] >=10]

# Normalize target
df["LogPrice"] = np.log(df["Price"])

as_cat_type = [
    "Type",
    "Furnishing_quality",
    "Condition",
    "Heating",
    "Energy_certificate",
    "Energy_certificate_type",
    "Energy_efficiency_class",
    "Garagetype",
]
df[as_cat_type] = df[as_cat_type].astype("category")

as_string_dtype = [
    "Energy_source",
    "State",
    "City",
    "Place",
]
for c in as_string_dtype:
    nan_mask = df[c].isna()
    df.loc[nan_mask, c] = np.nan
    df[c] = df[c].astype("string")

# Drop cols
df = df.drop(columns=[
    "Unnamed:_0",
    "Price",
    "Free_of_Relation",
])

# Drop duplicates ignoring the target
df = df.drop_duplicates(subset=[c for c in df.columns if c != task_mold.target_column_name])

df = df.sample(frac=1, random_state=42).reset_index(drop=True)
print("Loaded data shape:", df.shape)

Loaded data shape: (10317, 24)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 10,317
Columns: 24
Use sampling: False (sample size: 10,317)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['Place', 'Lot', 'Living_space', 'Energy_consumption', 'Usable_area', 'City', 'Year_built', 'Energy_source', 'Rooms', 'Year_renovated']
Rows remaining as candidates after top-10 filter: 148 (of 10,317)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,Type,Living_space,Lot,Usable_area,Rooms,Bedrooms,Bathrooms,Floors,Year_built,Furnishing_quality,Year_renovated,Condition,Heating,Energy_source,Energy_certificate,Energy_certificate_type,Energy_consumption,Energy_efficiency_class,State,City,Place,Garages,Garagetype,LogPrice
0,Mid-terrace house,200.0,2571.0,370.0,5.0,1.0,1.0,3.0,NaN,refined,NaN,maintained,underfloor heating,"Strom, Kohle, Holz",available,demand certificate,NaN,H,Sachsen,Mittelsachsen (Kreis),Lunzenau,4.0,Parking lot,10.799576
1,NaN,125.0,174.0,70.0,6.0,3.0,2.0,3.0,1990.0,normal,2015.0,renovated,NaN,l,NaN,NaN,NaN,NaN,Baden-Wrttemberg,Esslingen (Kreis),Nrtingen,2.0,Outside parking lot,13.122363
2,Residential property,90.0,693.0,15.0,4.0,2.0,1.0,NaN,1920.0,refined,NaN,as new,central heating,<NA>,not required by law,NaN,NaN,NaN,Bayern,Dachau (Kreis),Rhrmoos,NaN,NaN,13.498056
3,Mid-terrace house,255.0,752.0,NaN,6.0,4.0,3.0,2.0,2009.0,NaN,2009.0,fixer-upper,district heating,Strom,available,consumption certificate,28.9,A+,Nordrhein-Westfalen,Mrkischer Kreis,Ldenscheid,1.0,Garage,13.161584
4,Villa,624.0,2288.0,150.0,14.0,4.0,5.0,5.0,1904.0,NaN,2020.0,modernized,heat pump,Erdgas leicht,NaN,NaN,NaN,NaN,Baden-Wrttemberg,Baden-Baden,Innenstadt,1.0,Garage,14.580978


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,Energy_efficiency_class,category,4709.0,45.64,9.0,"D , F , H , E , G , C , B , A+ , A"
1,Energy_certificate_type,category,3443.0,33.37,2.0,"demand certificate, consumption certificate"
2,Furnishing_quality,category,2662.0,25.80,4.0,"normal, basic, refined, luxus"
3,Garagetype,category,1912.0,18.53,7.0,"Garage, Outside parking lot, Parking lot, Carport, Underground parking lot, Duplex lot, Car park lot"
4,Energy_certificate,category,721.0,6.99,3.0,"available, not required by law, available for inspection"
5,Heating,category,572.0,5.54,13.0,"stove heating, heat pump, central heating, oil heating, underfloor heating, night storage heater, district heating, wood-pellet heating, electric heating, floor heating"
6,Type,category,394.0,3.82,11.0,"Mid-terrace house, Duplex, Single dwelling, Farmhouse, Villa, Multiple dwelling, Residential property, Special property, Bungalow, Corner house"
7,Condition,category,304.0,2.95,10.0,"modernized, refurbished, dilapidated, maintained, renovated, fixer-upper, first occupation after refurbishment, first occupation, by arrangement, as new"
8,Energy_consumption,float64,7926.0,76.82,1420.0,"114.0, 128.0, 120.0, 130.0, 121.0, 119.0, 125.0, 97.0, 131.0, 87.0"
9,Year_renovated,float64,5081.0,49.25,67.0,"2019.0, 2018.0, 2017.0, 2020.0, 2015.0, 2016.0, 2010.0, 2014.0, 2012.0, 2013.0"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
Living_space,10317.0,216.075141,172.629078,16.32000,5600.00000
Lot,10317.0,1482.230280,8655.139414,0.00000,547087.00000
Usable_area,5443.0,134.258593,189.944754,0.00000,4034.00000
Rooms,10317.0,7.381991,5.391853,1.00000,170.00000
Bedrooms,6730.0,4.168499,2.588224,0.00000,61.00000
Bathrooms,8574.0,2.305342,1.744452,0.00000,44.00000
Floors,7733.0,2.282167,0.820104,0.00000,13.00000
Year_built,9638.0,1958.799958,55.875952,1300.00000,2022.00000
Year_renovated,5236.0,2010.707219,10.556295,1900.00000,2206.00000
Energy_consumption,2391.0,117.703133,54.086305,5.10000,503.94000


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column                  rank                                        
City                    1             Hannover (Kreis)    106   1.03
                        2           Rhein-Neckar-Kreis     94   0.91
                        3        Nordfriesland (Kreis)     93    0.9
                        4                Wetteraukreis     85   0.82
                        5             Rhein-Sieg-Kreis     76   0.74
Condition               1                   modernized   4352  42.18
                        2                  refurbished   1292  12.52
                        3                  dilapidated   1137  11.02
                        4                   maintained   1035  10.03
                        5                    renovated   1017   9.86
Energy_certificate      1                    available   6836  66.26
                        2          not required by law   1558   15.1
                        3     available for inspection   1202  11.65
                        4                         <NA>    721   6.99
Energy_certificate_type 1           demand certificate   4037  39.13
                        2                         <NA>   3443  33.37
                        3      consumption certificate   2837   27.5
Energy_efficiency_class 1                         <NA>   4709  45.64
                        2                           D     949    9.2
                        3                           F     847   8.21
                        4                           H     825    8.0
                        5                           E     767   7.43
Energy_source           1                         Gas    4435  42.99
                        2                           l    2533  24.55
                        3                         <NA>   1187  11.51
                        4                       Strom     539   5.22
                        5                    Fernwrme     175    1.7
Furnishing_quality      1                       normal   3668  35.55
                        2                         <NA>   2662   25.8
                        3                        basic   2617  25.37
                        4                      refined    904   8.76
                        5                        luxus    466   4.52
Garagetype              1                       Garage   4325  41.92
                        2                         <NA>   1912  18.53
                        3          Outside parking lot   1666  16.15
                        4                  Parking lot   1562  15.14
                        5                      Carport    714   6.92
Heating                 1                stove heating   5863  56.83
                        2                    heat pump    968   9.38
                        3              central heating    853   8.27
                        4                  oil heating    716   6.94
                        5                         <NA>    572   5.54
Place                   1                         <NA>    278   2.69
                        2                   Innenstadt     31    0.3
                        3                    Falkensee     23   0.22
                        4                   Stadtmitte     20   0.19
                        5                       Hameln     19   0.18
State                   1          Nordrhein-Westfalen   1635  15.85
                        2                       Bayern   1312  12.72
                        3             Baden-Wrttemberg   1280  12.41
                        4                Niedersachsen   1264  12.25
                        5              Rheinland-Pfalz    996   9.65
Type                    1            Mid-terrace house   4228  40.98
                        2                       Duplex   2083  20.19
                        3              Single dwelling   1148  11.13
                        4                    Farmhouse    565   5.48
                        5                      

In [8]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,0.0,-0.35,-0.649,0.719,0.005,log,99272.6,4.491623e+15,exponential


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended splits: n_repeats=3, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to immoscout_german_house_prices/019d635a-0de5-72fd-857c-da243744398f
019d635a-0de5-72fd-857c-da243744398f
fb670d686e541a43358c4b3bf46529794f067c9de492a99c1f13cb8cccefb8c9
